# Programming a P4 Switches
Programming a P4 switch, whether targeting hardware or software, requires a dedicated software development environment equipped with a compiler. 

**Figure 1: Generic workflow design**

![Workflow](./figs/p4-workflow.png)

The workflow consists of the following components:

1. **Vendor-Provided Components**:
   - **Compiler**: Responsible for translating the target-independent P4 source code into platform-specific implementations.
   - **Architecture Model**: Defines the capabilities and constraints of the target device.
   - **Target Switch**: The hardware or software environment where the data plane logic is deployed.

2. **User-Provided Component**:
   - **P4 Source Code**: Customized by the user to specify the desired forwarding logic and behavior.

### Compilation Process

The compiler translates the user-defined P4 source code into two essential artifacts:

1. **Data Plane Runtime Configuration**:  
   - Implements the forwarding logic defined in the P4 program.
   - Contains instructions and resource mappings tailored to the specific target switch.

2. **Runtime Application Programming Interfaces (APIs)**:  
   - Enable the control plane or user to interact with the data plane during runtime.
   - Support operations such as adding/removing entries in match-action tables and reading/writing the state of external objects (e.g., counters, meters, registers).
   - Provide the necessary information for the control plane to manipulate data plane components, including:
     - Table identifiers
     - Match fields
     - Keys
     - Action parameters

**References**: The [P4 SDE](https://p4.org/intels-tofino-p4-software-is-now-open-source/) has been open-sourced and can be accessed [here](https://github.com/p4lang/open-p4studio).

## Set Up a Network Slice with a P4 Tofino Switch and a Basic P4 Program

In this notebook, we will configure a network slice that utilizes a P4 Tofino switch to run a basic P4 program. The setup will establish connectivity between two nodes via the P4 switch, allowing them to communicate and test connectivity using `ping`.

This example is inspired by the **University of South Carolina CyberTraining program** for P4 programmable devices, which provides hands-on training on using P4 to build and manage network slices effectively.

  - [CyberTraining Lab](https://netlab.cec.sc.edu/)
  - [BMv2 Examples](https://learn.fabric-testbed.net/knowledge-base/p4-programmable-data-plane-switches-bmv2-over-fabric/)

## Prerequisites

1. **Access to a P4 Tofino switch**: A programmable hardware device equipped with an Intel Tofino ASIC.
2. **Two nodes for connectivity**: Ensure two nodes are physically or virtually connected to the P4 switch.
3. **Basic knowledge of P4 programming**: Familiarity with writing, compiling, and deploying P4 programs is necessary.
4. **Installed Tofino Software Development Environment (SDE)**: Required tools and libraries for P4 development and deployment.

## Example Workflow Overview

1. **Compile the P4 Program**:
   - Use the **bf-p4c compiler** with the **Tofino Native Architecture (TNA)** to compile the program into a binary file.
   - This binary will serve as the data plane runtime configuration for the P4 switch.

2. **Load the Data Plane Program**:
   - Load the compiled binary into the P4 switch daemon, ensuring the forwarding logic is active.

3. **Configure the Control Plane**:
   - Use the **bfrt-python API** to populate the match-action tables, specifying forwarding rules for the nodes.

4. **Test Connectivity**:
   - Use `ping` to test connectivity between the two nodes and validate that the network slice is functioning as intended.

**Figure 2: P4 example workflow**

![Lab Workflow](./figs/p4-lab-workflow.png)


**Figure 3: Slice Topology**

![Topology](./figs/p4-slice.png)

The topology depicts two nodes connected through the P4 switch. The P4 program running on the switch enables forwarding, allowing the nodes to communicate.


## Import the FABlib Library


In [1]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

fablib.show_config();

User: choueiri@email.sc.edu bastion key is valid!
Configuration is valid


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,8eaa3ec2-65e7-49a3-8c09-e1761141a6ad
Bastion Host,bastion.fabric-testbed.net
Bastion Username,choueiri_0000118746
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key


## Create the Experiment Slice

The following script sets up two nodes, each with Shared NICs connected to two ports on a P4 tofino switch.

NIC component model options include:
- NIC_Basic: 100 Gbps Mellanox ConnectX-6 SR-IOV VF (1 Port)
- NIC_ConnectX_5: 25 Gbps Dedicated Mellanox ConnectX-5 PCI Device (2 Ports)
- NIC_ConnectX_6: 100 Gbps Dedicated Mellanox ConnectX-6 PCI Device (2 Ports)

### Identify the Sites

List available sites with `P4 Switch`. Chooose one site at random from the available sites.

In [2]:
p4_column_name = 'p4-switch_available'

'''
# Find a site which has a P4 Switch available
[site2] = fablib.get_random_sites(count=1, filter_function=lambda x: x[p4_column_name] > 0)

# Choose another random site other than P4 site to host the VMs
site1 = fablib.get_random_site(avoid=[site2])
'''
site2="FIU"
site1="ATLA"

print(f"Sites chosen for hosting VMs: {site1} P4: {site2}")

Sites chosen for hosting VMs: ATLA P4: FIU


### Define variables

In [3]:
slice_name = 'P4-Lab-Slice2'
p4_column_name = "p4-switch_available"

node1_name = 'Node1'
node2_name = 'Node2'
p4_name = 'P4'
network1_name = 'net1'
network2_name = 'net2'

print(f"VM Site: {site1}")
print(f"P4 Site: {site2}")

model='NIC_ConnectX_6'
# model='NIC_Basic'

VM Site: ATLA
P4 Site: FIU


In [4]:
#Create Slice
slice = fablib.new_slice(name=slice_name)
node1_image = "default_ubuntu_20"
node2_image = "default_ubuntu_20"

# Create Network
net1 = slice.add_l2network(name=network1_name, subnet=IPv4Network("192.168.0.0/24"))
net2 = slice.add_l2network(name=network2_name, subnet=IPv4Network("192.168.0.0/24"))

# Create Node 1 and its links
node1 = slice.add_node(name=node1_name, site=site1, cores=16, ram=32, disk=200, image=node1_image)
iface1 = node1.add_component(model=model, name='nic1').get_interfaces()[0]
iface1.set_mode('config')
net1.add_interface(iface1)
iface1.set_ip_addr(IPv4Address("192.168.0.1"))

# Create P4 switch and its links 
p4 = slice.add_switch(name=p4_name, site=site2)
iface2 = p4.get_interfaces()[0]
iface3 = p4.get_interfaces()[1]

net1.add_interface(iface2)
net2.add_interface(iface3)

# Create Node 2 and its links
node2 = slice.add_node(name=node2_name, site=site1, cores=16, ram=32, disk=200, image=node2_image)
iface4 = node2.add_component(model=model, name='nic1').get_interfaces()[0]
iface4.set_mode('config')
net2.add_interface(iface4)
iface4.set_ip_addr(IPv4Address("192.168.0.2"))

# Submit Slice Request
slice.submit()


Retry: 30, Time: 1198 sec


ID,04fcd879-b6ff-4d50-84de-dec20a4ceb4c
Name,P4-Lab-Slice2
Lease Expiration (UTC),2026-06-10 18:02:29 +0000
Lease Start (UTC),2026-05-27 18:02:29 +0000
Project ID,8eaa3ec2-65e7-49a3-8c09-e1761141a6ad
State,StableOK
Email,CHOUEIRI@email.sc.edu
UserId,78504735-b34f-42a8-be20-8e71d6acf13e


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
b46b9e88-bdce-41cd-85f6-e63020e16147,Node1,16,32,500,default_ubuntu_20,qcow2,atla-w1.fabric-testbed.net,ATLA,ubuntu,2001:400:a100:3050:f816:3eff:feb0:4dba,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3050:f816:3eff:feb0:4dba,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
a20aac2d-af41-48e9-9eb6-44ad216fb84a,Node2,16,32,500,default_ubuntu_20,qcow2,atla-w1.fabric-testbed.net,ATLA,ubuntu,2001:400:a100:3050:f816:3eff:fe1f:5a6c,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3050:f816:3eff:fe1f:5a6c,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
a9a98a3b-9fae-477d-be51-c49c7974c965,P4,,,,,,,FIU,fabric,131.94.57.14,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config fabric@131.94.57.14,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
3a7b4196-46b7-415d-93af-8e3d91f79b44,net1,L2,L2STS,None,192.168.0.0/24,None,Active,
775074d4-e566-4094-9c37-504f4d860107,net2,L2,L2STS,None,192.168.0.0/24,None,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1-nic1-p1,p1,Node1,net1,100,config,,10:70:FD:E5:CD:F0,enp7s0,enp7s0,192.168.0.1,6,HundredGigE0/0/0/4
Node1-nic1-p2,p2,Node1,None,100,config,,10:70:FD:E5:CD:F1,enp8s0,enp8s0,None,6,None
p1,p1,P4,net1,100,config,,None,None,1,None,None,HundredGigE0/0/0/4
p2,p2,P4,net2,100,config,,None,None,2,None,None,HundredGigE0/0/0/24/1
p3,p3,P4,None,100,config,,None,None,3,None,None,None
p4,p4,P4,None,100,config,,None,None,4,None,None,None
p5,p5,P4,None,100,config,,None,None,5,None,None,None
p6,p6,P4,None,100,config,,None,None,6,None,None,None
p7,p7,P4,None,100,config,,None,None,7,None,None,None
p8,p8,P4,None,100,config,,None,None,8,None,None,None



Time to print interfaces 1202 seconds


'04fcd879-b6ff-4d50-84de-dec20a4ceb4c'

## Upload the P4 Code to the switch

In [ ]:
slice = fablib.get_slice(slice_name)
switch = slice.get_node(p4_name)
switch.upload_directory("P4_labs", ".")

## Compile the basic P4 program

### Configure the environment variables

SDE environment can be setup using the command:

```
sde-env-9.13.3
```

Example output is shown below:
```
fabric@P4:~$ sde-env-9.13.3

Intel Tofino SDE 9.13.3 on platform "accton_wedge100bf_32x"
Load/unload kernel modules: $ sudo $(type -p bf_{kdrv,kpkt,knet}_mod_{load,unload})

Compile: $ p4_build.sh <p4name>.p4
Run:     $ run_switchd.sh -p <p4name>

Build artifacts and logs are stored in /home/fabric/.bf-sde/9.13.3

Use "exit" or CTRL-D to exit this shell.
```
### Compile the code

In this example we will not modify the P4 code. Instead, we will just compile it and run it on the switch. 

The script `p4_build.sh` is provided by Intel’s support portal. It invokes the compiler to generate the output files. It automatically detects the P4 version (i.e., P416), and generates
the output files under `~/.bf-sde/9.13.3/build/<program_name>`, where `<program_name>` is the file `basic.p4`. It also generates the log files under `~/.bf-sde/9.13.3/logs/<program_name>` and other files (e.g., graphs).

Use the command below to compile the code:

```
p4_build.sh ~/P4_labs/lab1/p4src/basic.p4
```

After executing the command, if there are no error messages displayed in the terminal, then the P4 program was compiled successfully.

In [ ]:
# Execute a series of commands on the switch using execute()
stdout, stderr = switch.execute(command=[

    # Step 1: Enter the Intel Tofino SDE environment
    # This command activates the environment needed to run P4-related tools.
    ("sde-env-9.13.3", r"\[nix\-shell\(SDE\-9.13.3\):.*\$ ", 10),

    # Step 2: Compile the P4 program using p4_build.sh
    # This builds the P4 program specified in the path (basic.p4).
    # The build process may take some time, so a timeout of 20 seconds is used.
    ("p4_build.sh P4_labs/lab1/p4src/basic.p4", r"\[nix\-shell\(SDE-9.13.3\):.*\$ ", 20),

    # Step 3: Exit the SDE environment cleanly
    # Ensures that the environment is properly terminated after execution.
    ("exit", r"\[nix\-shell\(SDE-9.13.3\):.*\$ ", 10)
])

# stdout: Captures the standard output of the executed commands.
# stderr: Captures any error messages encountered during execution.


### Verify the compilation output files
Verify the output files for the program basic.p4 are generated in the `~/.bf-sde/9.13.3/build/<program_name>` directory by issuing the following command.

```
ls ~/.bf-sde/9.13.3/build/basic/
```

The binary file that corresponds to the compiled data plane is located under `ls ~/.bf-sde/9.13.3/build/<program_name>/tofino/pipe`. Use the command below to display the contents of this directory.

```
ls ~/.bf-sde/9.13.3/build/basic/tofino/pipe/
```

In [ ]:
stdout, stderr = switch.execute("ls ~/.bf-sde/9.13.3/build/basic/")

In [ ]:
stdout, stderr = switch.execute("ls ~/.bf-sde/9.13.3/build/basic/tofino/pipe/")

## Start the switch daemon and configure the switch ports

Now that we have compiled our P4 program and generated the output files. We can start the switch daemon with the compiled output using the command below.


Load the  kernel module, and start the switch daemon using the following commands:
```
sde-env-9.13.3

sudo $SDE_INSTALL/bin/./bf_kdrv_mod_load $SDE_INSTALL

run_switchd.sh -p basic
```

Once the daemon is running, you can issue commands in the bfshell.  

1. Issue the following command in the switch CLI to manage the ports of the switch.

```
ucli
```

NOTE: The ucli is the bfshell instance used to manage the switch ports. With the ucli, you can enable or disable ports, set the port speed (e.g., 100 Gbps, 40 Gbps, and 10 Gbps), and
select the FEC type. Additionally, the user can monitor the status of the ports, the number of sent and received frames, and other variables.

2. Now we will add the ports for the switch. Recall from the topology that Server1 is connected to port 1 on the switch, and Server2 is connected to port 2 on the switch. Issue the
commands below to add ports 1 and 2 in the Tofino switch.

```
pm port-add 1/- 100G NONE
pm port-add 2/- 100G NONE
```

3. Enable the ports by issuing the following commands.
```
pm port-enb 1/-
pm port-enb 2/-
```

4. Verify that the ports are up by issuing the following command.
```
pm show
```


**Expected output:**
```
bf-shell> ucli
bf-sde> pm port-add 1/- 100G NONE
bf-sde> pm port-add 2/- 100G NONE
bf-sde> pm show
-----+----+---+----+-------+----+--+--+---+---+---+--------+----------------+----------------+-
PORT |MAC |D_P|P/PT|SPEED  |FEC |AN|KR|RDY|ADM|OPR|LPBK    |FRAMES RX       |FRAMES TX       |E
-----+----+---+----+-------+----+--+--+---+---+---+--------+----------------+----------------+-
1/0  |23/0|132|3/ 4|100G   |NONE|Au|Au|YES|ENB|DWN|  NONE  |               0|               0|
2/0  |22/0|140|3/12|100G   |NONE|Au|Au|YES|ENB|DWN|  NONE  |               0|               0|
bf-sde> pm port-enb 1/-
bf-sde> pm port-enb 2/-
bf-sde> pm show
-----+----+---+----+-------+----+--+--+---+---+---+--------+----------------+----------------+-
PORT |MAC |D_P|P/PT|SPEED  |FEC |AN|KR|RDY|ADM|OPR|LPBK    |FRAMES RX       |FRAMES TX       |E
-----+----+---+----+-------+----+--+--+---+---+---+--------+----------------+----------------+-
1/0  |23/0|132|3/ 4|100G   |NONE|Au|Au|YES|ENB|UP |  NONE  |              29|              14|
2/0  |22/0|140|3/12|100G   |NONE|Au|Au|YES|ENB|UP |  NONE  |              29|              14|
```

The ouput above shows that ports 1 and 2 are up. The speed for the ports is 100G. The columns `FRAMES TX` indicate how many frames are received and `FRAMES RX` transmitted in each port respectively.

**NOTE**: These commands can also be executed manually by accessing the P4 switch via SSH.

In [ ]:
thread = switch.execute_thread(command=[
    # Enter SDE environment
    ("sde-env-9.13.3", r"\[nix\-shell\(SDE\-9.13.3\):.*\$ ", 10),

    # Load Kernel Modules
    ("sudo $SDE_INSTALL/bin/./bf_kdrv_mod_load $SDE_INSTALL", r"\[nix\-shell\(SDE-9.13.3\):.*\$ ", 20),

    # Start run_switchd.sh interactively (DO NOT run in background)
    ("run_switchd.sh -p basic", r"bfshell>", 30),  # Wait for switchd prompt to appear

    # Enter UCLI after switchd starts
    ("ucli", r"bf-sde>", 10),

    # Port configuration inside UCLI
    ("pm port-add 1/- 100G NONE", r"bf-sde>", 10),
    ("pm port-add 2/- 100G NONE", r"bf-sde>", 10),
    ("pm port-enb 1/-", r"bf-sde>", 10),
    ("pm port-enb 2/-", r"bf-sde>", 10),
    ("pm show", r"bf-sde>", 10),
    ("pm show", r"bf-sde>", 10),
    ("pm show", r"bf-sde>", 10),
    # Keep the session open to prevent exit
    ("sleep infinity", r"bf-sde>", 300)
], output_file="run_switchd.log")

In [ ]:
import time
time.sleep(120)

## Populating the switch’s forwarding table

**1. Update the port numbers highligted in red box in `~/P4_labs/lab1/bfrt_python/setup.py`. Port numbers should match the `D_P` column as observed in the output of `pm show` above.**

<img src="./figs/p4-brft-setup.png" alt="brft" style="width:40%;">


**NOTE**: These steps can also be executed manually by accessing the P4 switch via SSH.

In [ ]:
file_path = "~/P4_labs/lab1/bfrt_python/setup.py"
INGRESS = switch.get_interfaces()[0].get_device_name()
EGRESS = switch.get_interfaces()[1].get_device_name()

switch.execute(
    f"sed -i 's/ingress_logical *= *[0-9]\\+/ingress_logical = {INGRESS}/' {file_path}"
)
switch.execute(
    f"sed -i 's/egress_logical *= *[0-9]\\+/egress_logical = {EGRESS}/' {file_path}"
)


**2. Populate the table entries to the switch by typing the following command in the embedded terminal.**

```
sde-env-9.13.3

run_bfshell.sh --no-status-srv -b ~/P4_labs/lab1/bfrt_python/setup.py
```

The forwarding table contains the information that the P4 program will use to forward packets to the right destination. 

**NOTE**: Please wait to run this step until you see ports as shown below to ensure the port have been enabled by the previous step. If you see an error, please re-run this step.

**NOTE**: These steps can also be executed manually by accessing the P4 switch via SSH.

```
-----+----+---+----+-------+----+--+--+---+---+---+--------+----------------+----------------+-
PORT |MAC |D_P|P/PT|SPEED  |FEC |AN|KR|RDY|ADM|OPR|LPBK    |FRAMES RX       |FRAMES TX       |E
-----+----+---+----+-------+----+--+--+---+---+---+--------+----------------+----------------+-
1/0  |23/0|132|2/ 4|100G   |NONE|Au|Au|YES|ENB|UP |  NONE  |               7|               0| 
2/0  |22/0|140|2/12|100G   |NONE|Au|Au|YES|ENB|UP |  NONE  |               7|               0|
```

In [ ]:
# Execute a series of commands on the switch using execute()
stdout, stderr = switch.execute(command=[
    
    # Step 1: Enter the Intel Tofino SDE environment
    # This command activates the environment needed to run P4-related tools.
    ("sde-env-9.13.3", r"\[nix\-shell\(SDE\-9.13.3\):.*\$ ", 10),

    # Step 2: Run the BFShell script to set up the P4 runtime environment
    # This script initializes the environment and configures the necessary settings.
    # --no-status-srv: Disables the status server.
    # -b <script>: Runs the Python setup script inside bfshell.
    ("run_bfshell.sh --no-status-srv -b ~/P4_labs/lab1/bfrt_python/setup.py", 
     r"\[nix\-shell\(SDE\-9.13.3\):.*\$ ", 10),

    # Step 3: Exit the SDE environment cleanly
    # Ensures that the environment is properly terminated after execution.
    #("exit", r"\[nix\-shell\(SDE\-9.13.3\):.*\$ ", 10)
])

# stdout will contain the standard output from the executed commands.
# stderr will contain any error messages encountered during execution.


## Verifying Reachability through Ping Tests

Initiate a ping between the VMs.  

**NOTE:** The ping will function as long as the switch daemon is actively running on the Tofino switch and is correctly configured.  

In this example, the switch daemon automatically terminates after **5 minutes**, which may cause the ping to stop working beyond this duration. This is expected behavior. To resume pinging, the user must restart the switch daemon.

In [ ]:
slice=fablib.get_slice(slice_name)
node1=slice.get_node(node1_name)
node2=slice.get_node(node2_name)

node1_addr = node1.get_interface(network_name=network1_name).get_ip_addr()
node2_addr = node2.get_interface(network_name=network2_name).get_ip_addr()

stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')
stdout, stderr = node2.execute(f'ping -c 5 {node1_addr}')

## Delete the Slice

Please delete your slice when you are done with your experiment.

In [ ]:
slice=fablib.get_slice(slice_name)
slice.delete()